# Module 5.4: When Training Goes Wrong (A Debugging Clinic)

Your capstone in Module 5.3 trained on the first try — because the code was already debugged for you. Real projects never go like that. The loss gets stuck, or explodes, or (worst of all) *looks great while the model learns nothing*.

In this module we take the same tiny GPT and **break its training on purpose, four classic ways**. For each failure you'll learn its *fingerprint* (what you see), its *prime suspect* (what usually causes it), and the *one-line check* that confirms the diagnosis.

Bookmark this table — it's the whole module:

| Symptom | Prime suspect | First check |
|---|---|---|
| Loss **stuck** at ≈ ln(vocab_size) | nothing is updating: optimizer holds the wrong params, lr=0, weights frozen | do the weights actually change after `step()`? |
| Loss falls **suspiciously fast** toward 0 | data leak: targets aren't shifted, or the model can see its target | print one `(x, y)` pair; sample from the model |
| Loss **climbs / becomes NaN** | learning rate too high, exploding gradients | log the gradient norm; lower lr; clip |
| **Crash**: shape mismatch in the loss | logits/targets not flattened for cross-entropy | read the error: CE wants `(N, C)` vs `(N,)` |
| Train loss ↓ but **val loss ↑** | overfitting | Module 5.6 (Evaluation) covers this one |
| Loss ↓ but **generated text is garbage** | broken causal mask or positions | an invariant test, like the KV-cache check in `tests/test_model.py` |

## 0. The Patient: a tiny GPT and a healthy baseline

Same setup as the capstone, shrunk for speed. One number to memorize before anything else: a model guessing **uniformly at random** over a vocabulary of $V$ tokens has loss $\ln(V)$. For our ~65-character vocabulary that's $\ln(65) \approx 4.17$. Every diagnosis below starts by comparing the loss to that baseline.

In [1]:
import os
import torch
import torch.nn.functional as F

from llm_workout.model import GPT

torch.manual_seed(0)
device = 'cpu'   # tiny model -- CPU is plenty, and keeps every run reproducible

# Data: local TinyShakespeare (downloaded by Module 5.3), tiny fallback otherwise.
if os.path.exists('tinyshakespeare.txt'):
    text = open('tinyshakespeare.txt', encoding='utf-8').read()
else:
    text = ("First Citizen:\nBefore we proceed any further, hear me speak.\n"
            "All:\nSpeak, speak.\n") * 500

chars = sorted(set(text))
vocab_size = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]

data = torch.tensor(encode(text), dtype=torch.long)
batch_size, block_size = 16, 32

def get_batch():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])  # shifted by 1!
    return x, y

def make_model():
    torch.manual_seed(0)
    return GPT(vocab_size=vocab_size, d_model=64, num_layers=2,
               num_heads=2, hidden_dim=256, max_seq_len=64)

def train(model, batch_fn, steps=200, lr=3e-4, log_every=50):
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for step in range(steps):
        xb, yb = batch_fn()
        _, loss, _ = model(xb, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            print(f"  step {step:3d} | loss {loss.item():.4f}")
    return loss.item()

import math
print(f"Random-guessing baseline: ln({vocab_size}) = {math.log(vocab_size):.4f}\n")
print("A HEALTHY run looks like this (loss starts near the baseline, then falls):")
healthy = train(make_model(), get_batch)

Random-guessing baseline: ln(65) = 4.1744

A HEALTHY run looks like this (loss starts near the baseline, then falls):


  step   0 | loss 4.1588


  step  50 | loss 3.4046


  step 100 | loss 3.0374


  step 150 | loss 2.8299


  step 199 | loss 2.7125


## 1. Failure: "My loss is AMAZING" (too good to be true)

### The Scenario
You fumble `get_batch` and return the targets **unshifted** — `y` is the *same slice* as `x` instead of the slice moved one step right. At every position the "next token to predict" is... the token the model can already see. The task collapses into *copying*.

### The Fingerprint
The loss doesn't just fall — it **plummets toward 0**, far below anything a real language model achieves on real text. (For reference, your healthy run above is still around 2.5 after 200 steps.) If your loss looks like a miracle, it's a leak.

In [2]:
def get_batch_leaky():
    ix = torch.randint(len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i:i + block_size] for i in ix])   # BUG: same slice, no shift!
    return x, y

print("Training with the leaky batch function:")
leaky_loss = train(make_model(), get_batch_leaky)
print(f"\nFinal loss {leaky_loss:.3f} -- 'better' than the healthy run's {healthy:.3f}.")
print("New state of the art?  No. The model learned to COPY, not to predict.")

Training with the leaky batch function:
  step   0 | loss 3.2733


  step  50 | loss 2.2242


  step 100 | loss 1.4508


  step 150 | loss 0.8610


  step 199 | loss 0.5282

Final loss 0.528 -- 'better' than the healthy run's 2.712.
New state of the art?  No. The model learned to COPY, not to predict.


### The One-Line Check
Decode a single `(x, y)` pair and *look at it*. In a correct batch, `y` is `x` shifted left by one. In the leaky batch, they're identical — the giveaway.

In [3]:
itos = {i: ch for ch, i in stoi.items()}
decode = lambda t: ''.join(itos[int(i)] for i in t)

xg, yg = get_batch()
xb, yb = get_batch_leaky()
print("HEALTHY  x[0]:", repr(decode(xg[0][:20])))
print("HEALTHY  y[0]:", repr(decode(yg[0][:20])), "  <- same text, one step ahead")
print()
print("LEAKY    x[0]:", repr(decode(xb[0][:20])))
print("LEAKY    y[0]:", repr(decode(yb[0][:20])), "  <- IDENTICAL. Busted.")
print("\nLesson: the one-position shift IS the whole task (Module 5.1).")
print("Whenever a loss looks miraculous, print a batch before you celebrate.")

HEALTHY  x[0]: 'orth walked on his w'
HEALTHY  y[0]: 'rth walked on his wa'   <- same text, one step ahead

LEAKY    x[0]: ' beget!\nO boy, thy f'
LEAKY    y[0]: ' beget!\nO boy, thy f'   <- IDENTICAL. Busted.

Lesson: the one-position shift IS the whole task (Module 5.1).
Whenever a loss looks miraculous, print a batch before you celebrate.


## 2. Failure: the loss is a flat line at 4.17

### The Scenario
A subtler classic: you refactor, and the optimizer ends up holding **a different model's parameters** — a stale copy, a re-instantiated model, the un-wrapped version of a wrapped model. Everything *runs*. Gradients are computed. `opt.step()` dutifully updates... the wrong tensors.

### The Fingerprint
The loss sits at the random baseline forever — it doesn't fall, it doesn't explode, it just *sits there*, pinned at $\ln(V)$.

In [4]:
model = make_model()
shadow = make_model()     # an innocent-looking second instance (a "stale copy")

opt = torch.optim.AdamW(shadow.parameters(), lr=3e-4)   # BUG: wrong model's params!

print("Training `model` while the optimizer holds `shadow`'s parameters:")
for step in range(100):
    xb, yb = get_batch()
    _, loss, _ = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 25 == 0 or step == 99:
        print(f"  step {step:3d} | loss {loss.item():.4f}   <- pinned at the baseline")

Training `model` while the optimizer holds `shadow`'s parameters:
  step   0 | loss 4.1588   <- pinned at the baseline
  step  25 | loss 4.1721   <- pinned at the baseline


  step  50 | loss 4.1844   <- pinned at the baseline
  step  75 | loss 4.1850   <- pinned at the baseline


  step  99 | loss 4.1898   <- pinned at the baseline


### The One-Line Check
Snapshot a weight, take one optimizer step, and ask: **did anything move?** (And directly: *are the tensors the optimizer holds the same objects as the model's?*)

In [5]:
before = model.token_embedding.weight.detach().clone()
xb, yb = get_batch()
_, loss, _ = model(xb, yb)
opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
after = model.token_embedding.weight.detach()

print(f"Did the model's weights change after step()?  {not torch.equal(before, after)}")

model_params = set(id(p) for p in model.parameters())
opt_params = set(id(p) for group in opt.param_groups for p in group['params'])
print(f"Does the optimizer hold the model's tensors?  {bool(model_params & opt_params)}")

print("\nBoth 'False' -> the optimizer is updating something else entirely.")
print("Fix: opt = torch.optim.AdamW(model.parameters(), lr=3e-4)  -- and re-check.")
print("(The same fingerprint appears with lr=0, or params frozen by requires_grad=False.)")

Did the model's weights change after step()?  False
Does the optimizer hold the model's tensors?  False

Both 'False' -> the optimizer is updating something else entirely.
Fix: opt = torch.optim.AdamW(model.parameters(), lr=3e-4)  -- and re-check.
(The same fingerprint appears with lr=0, or params frozen by requires_grad=False.)


## 3. Failure: the loss climbs, then prints `nan`

### The Scenario
You get impatient and crank the learning rate. Each step now *overshoots* the valley it's aiming for, landing higher up the opposite wall. Higher loss → steeper slope → even bigger next step. The spiral ends in `inf`, then `nan` — and once a single `nan` enters the weights, it infects everything it touches.

### The Fingerprint
Loss rises instead of falling, and (with raw SGD) hits `inf`, then `nan`, within a handful of steps. The **gradient norm** — one number summarizing how big the update pressure is — blows up first. Log it; it's your early-warning siren.

(We use plain SGD here because its raw, unscaled steps make the explosion vivid. AdamW's per-weight scaling can *mask* a too-hot lr — the loss climbs and thrashes instead of cleanly `nan`-ing — which makes it *harder* to notice. Same disease, quieter symptoms.)

In [6]:
model = make_model()
opt = torch.optim.SGD(model.parameters(), lr=10.0)   # BUG: catastrophically hot

print("SGD lr=10 -- watch the gradient norm explode first, then the loss follow:")
for step in range(8):
    xb, yb = get_batch()
    _, loss, _ = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=float('inf'))
    opt.step()
    print(f"  step {step} | loss {loss.item():10.4f} | grad norm {grad_norm.item():12.4f}")

SGD lr=10 -- watch the gradient norm explode first, then the loss follow:
  step 0 | loss     4.1588 | grad norm       2.0616
  step 1 | loss    13.3233 | grad norm      44.6267
  step 2 | loss  1592.5713 | grad norm    1103.3101
  step 3 | loss 219972.6562 | grad norm   10963.0352
  step 4 | loss 1902573568.0000 | grad norm 29379952.0000
  step 5 | loss     4.1744 | grad norm          nan
  step 6 | loss        nan | grad norm          nan
  step 7 | loss        nan | grad norm          nan


In [7]:
# Damage control: GRADIENT CLIPPING rescales any update whose norm exceeds a cap.
# Real LLM training runs almost always clip (max_norm=1.0 is the common default).
model = make_model()
opt = torch.optim.SGD(model.parameters(), lr=10.0)   # same terrible lr...

print("Same SGD lr=10, but with clip_grad_norm_(max_norm=1.0):")
for step in range(8):
    xb, yb = get_batch()
    _, loss, _ = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)   # the seatbelt
    opt.step()
    print(f"  step {step} | loss {loss.item():10.4f}")

print("\nNo nan -- but the loss still isn't really falling. Clipping is a seatbelt")
print("for gradient SPIKES, not a cure for a wrong learning rate. Fix the lr too.")

Same SGD lr=10, but with clip_grad_norm_(max_norm=1.0):
  step 0 | loss     4.1588
  step 1 | loss     8.7059
  step 2 | loss     9.7495
  step 3 | loss     5.5832
  step 4 | loss     5.0619
  step 5 | loss     7.7766
  step 6 | loss     5.9313
  step 7 | loss    16.2985

No nan -- but the loss still isn't really falling. Clipping is a seatbelt
for gradient SPIKES, not a cure for a wrong learning rate. Fix the lr too.


## 4. Failure: the shape-mismatch crash

### The Scenario
The most common *crash* in all of LLM training. Your model outputs logits shaped `(batch, seq_len, vocab)` — but `F.cross_entropy` wants `(N, classes)` against targets `(N,)`. Until you flatten batch and time into one long list of predictions, it refuses.

Learning to *read* this error beats memorizing the fix.

In [8]:
model = make_model()
xb, yb = get_batch()
logits, _, _ = model(xb)
print(f"logits: {tuple(logits.shape)}   targets: {tuple(yb.shape)}\n")

try:
    F.cross_entropy(logits, yb)          # BUG: 3-D logits, 2-D targets
except RuntimeError as e:
    print(f"RuntimeError: {e}\n")

# The error says CE got shapes it can't pair up. The fix: flatten (batch, seq)
# into one axis -- every position is just "a prediction" as far as CE cares.
loss = F.cross_entropy(logits.view(-1, logits.size(-1)),   # (B*T, vocab)
                       yb.view(-1))                        # (B*T,)
print(f"After flattening: loss = {loss.item():.4f}  (this is exactly what")
print("the library's GPT.forward does internally -- see src/llm_workout/model.py)")

logits: (16, 32, 65)   targets: (16, 32)

RuntimeError: Expected target size [16, 65], got [16, 32]

After flattening: loss = 4.1588  (this is exactly what
the library's GPT.forward does internally -- see src/llm_workout/model.py)


## 5. The Positive Tool: the single-batch overfit test

Debugging isn't only about failures — you also want a quick *"is my pipeline fundamentally sound?"* test. Here's the standard one:

> **A correct model + loss + optimizer must be able to drive the loss to ~0 on a single, fixed batch.** It's just memorization. If your setup can't even memorize 16 examples, there is a *bug* — not a capacity problem, not a data problem, a bug.

Run this before every long training run. It takes seconds and catches whole categories of wiring mistakes.

In [9]:
model = make_model()
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)

xb, yb = get_batch()          # ONE fixed batch, reused every step
for step in range(301):
    _, loss, _ = model(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"  step {step:3d} | loss on the SAME batch: {loss.item():.4f}")

print("\nLoss collapsing toward 0 on one batch = the machinery is sound.")
print("(If you saw this stall at 4.17 instead, revisit Section 2.)")

  step   0 | loss on the SAME batch: 4.1588


  step 100 | loss on the SAME batch: 0.5309


  step 200 | loss on the SAME batch: 0.0789


  step 300 | loss on the SAME batch: 0.0372

Loss collapsing toward 0 on one batch = the machinery is sound.
(If you saw this stall at 4.17 instead, revisit Section 2.)


## Summary — the debugging ritual

When training misbehaves, walk this list *in order*:

1. **Know your baseline.** Compute $\ln(V)$ before you train. Stuck there = nothing is learning (§2). Far below it, instantly = something is leaking (§1).
2. **Look at one batch.** Decode `x[0]` and `y[0]` and read them with your eyes (§1).
3. **Check that weights move.** Snapshot → step → compare (§2).
4. **Log the gradient norm.** Explosions announce themselves there first; clip as a seatbelt (§3).
5. **Read shape errors slowly.** They tell you the two shapes; your job is only to explain the mismatch (§4).
6. **Overfit a single batch** before every serious run (§5).
7. **Sample early, sample often.** Loss is one number; generated text is the ground truth of what the model does. (The capstone printed samples *during* training for exactly this reason.)

Next stops: **Module 5.5** — controlling generation (decoding & sampling) — and **Module 5.6** — measuring whether the model is actually any good.

### 🏋️ Try it yourself

1. **The forgotten `zero_grad`.** Delete `opt.zero_grad(...)` from the healthy training loop and retrain. Gradients now *accumulate* across steps (Module 5.2). What's the fingerprint — smooth learning, noise, or explosion? Log the grad norm to confirm your theory.
2. **Instrument your capstone.** Go back to Module 5.3 and add gradient-norm logging (`clip_grad_norm_(..., float('inf'))` returns the norm without clipping) to its training loop. What's a *normal* norm for that model, and how would you notice a bad run early?
3. **Device mismatch.** If you have a GPU (or Apple `mps`): move the *model* to it but keep the batch on CPU, and run one forward pass. Read the error carefully — it names both devices. This is the #1 crash when graduating from toy to real hardware.

In [10]:
# Starter for Task 1: the healthy loop with zero_grad removed.
model = make_model()
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)

for step in range(100):
    xb, yb = get_batch()
    _, loss, _ = model(xb, yb)
    # opt.zero_grad(set_to_none=True)   # <-- "forgotten"
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float('inf'))
    opt.step()
    if step % 25 == 0 or step == 99:
        print(f"  step {step:3d} | loss {loss.item():.4f} | grad norm {grad_norm.item():.2f}")

# Your diagnosis here: what is accumulating, and what does it do to the steps?

  step   0 | loss 4.1588 | grad norm 2.06


  step  25 | loss 3.6695 | grad norm 32.46


  step  50 | loss 3.4543 | grad norm 53.41


  step  75 | loss 3.3242 | grad norm 62.65
  step  99 | loss 3.2442 | grad norm 65.46
